# Chapter 3 — TensorFlow, PyTorch, JAX & Keras

Maps to Chollet Ch.3. You now know *what* a net does (Ch.2). This chapter is *how to express it in code*:
the framework stack, the low-level TensorFlow primitives (so custom loops aren't magic), and the
**Keras API** you'll actually write models in.

### The one mental model
- **Keras** = high-level "building kit" — layers, models, `compile`, `fit`. You live here.
- **TensorFlow / PyTorch / JAX** = low-level "raw materials" — tensors, autodiff, GPU. Keras runs *on top* of one of them (the **backend**).
- All three backends give the same 3 things: **autodiff**, **GPU tensor math**, **multi-device**.

Pick a backend once, before importing keras:

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"   # "tensorflow" | "jax" | "torch"
import keras
print("keras", keras.__version__, "| backend:", keras.backend.backend())


## 1. TensorFlow primitives (the engine room)
You rarely write these directly, but you must read them — custom training loops and debugging need them.

In [ ]:
import tensorflow as tf

# constant tensors (like numpy, but immutable)
print(tf.ones((2,1)).numpy().ravel(), tf.zeros((2,1)).numpy().ravel())
print(tf.constant([1,2,3], dtype="float32").numpy())
print(tf.random.normal((2,2)).numpy().round(2))

# tensors are NOT assignable: x[0,0]=0 -> ERROR. Use a Variable for trainable state.
v = tf.Variable(tf.zeros((3,)))
v.assign(tf.ones((3,)))      # set
v.assign_add(tf.ones((3,)))  # += , also assign_sub for -=
v[0].assign(9.0)             # element assign
print("variable:", v.numpy())


In [ ]:
# ops mirror numpy: tf.square, tf.sqrt, tf.matmul, +, tf.concat ...
a = tf.ones((2,2)); print((tf.matmul(a, tf.square(a))).numpy())
# a Dense layer is exactly this:
def dense(x, W, b): return tf.nn.relu(tf.matmul(x, W) + b)
print(dense(tf.ones((1,3)), tf.ones((3,4)), tf.zeros((4,))).numpy())


### GradientTape — the thing NumPy can't do
Record ops in a `with tf.GradientTape()` scope, then ask for `gradient(output, inputs)`.
Variables are watched automatically; constants need `tape.watch(x)`.

In [ ]:
x = tf.Variable(3.0)
with tf.GradientTape() as tape:
    y = tf.square(x)          # y = x^2
print("d(x^2)/dx at 3 =", tape.gradient(y, x).numpy())   # = 2x = 6

# constant input must be watched explicitly
xc = tf.constant(3.0)
with tf.GradientTape() as tape:
    tape.watch(xc)
    y = tf.square(xc)
print("watched const grad =", tape.gradient(y, xc).numpy())

# nested tapes -> second-order gradient (acceleration of pos=4.9 t^2 is 9.8)
t = tf.Variable(1.0)
with tf.GradientTape() as outer:
    with tf.GradientTape() as inner:
        pos = 4.9 * t**2
    speed = inner.gradient(pos, t)
print("acceleration =", outer.gradient(speed, t).numpy())


### `@tf.function` — compile for speed
Eager code runs op-by-op (great for debugging, slow). Decorating fuses/compiles it. Debug eager first,
then add the decorator. `jit_compile=True` uses XLA (often faster still).

In [ ]:
@tf.function(jit_compile=True)
def dense_fast(x, W, b):
    return tf.nn.relu(tf.matmul(x, W) + b)
print(dense_fast(tf.ones((1,3)), tf.ones((3,4)), tf.zeros((4,))).numpy())


## 2. End-to-end in *pure* TF: a linear classifier (the interview classic)
Everything from scratch — no Keras model. `prediction = X @ W + b`, MSE loss, manual gradient step.
You should understand every line.

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
N = 1000
neg = rng.multivariate_normal(mean=[0,3], cov=[[1,.5],[.5,1]], size=N)
pos = rng.multivariate_normal(mean=[3,0], cov=[[1,.5],[.5,1]], size=N)
inputs  = np.vstack((neg, pos)).astype("float32")               # (2000, 2)
targets = np.vstack((np.zeros((N,1)), np.ones((N,1)))).astype("float32")  # (2000,1)

W = tf.Variable(tf.random.uniform((2,1)))
b = tf.Variable(tf.zeros((1,)))
lr = 0.1

@tf.function
def training_step(X, y):
    with tf.GradientTape() as tape:
        pred = tf.matmul(X, W) + b
        loss = tf.reduce_mean(tf.square(y - pred))
    gW, gb = tape.gradient(loss, [W, b])
    W.assign_sub(lr*gW); b.assign_sub(lr*gb)
    return loss

for step in range(40):
    loss = training_step(inputs, targets)
print(f"final loss: {float(loss):.4f}  | W={W.numpy().ravel().round(3)} b={float(b):.3f}")


In [ ]:
import matplotlib.pyplot as plt
pred = (tf.matmul(inputs, W) + b).numpy()[:,0]
xs = np.linspace(-2, 5, 100)
boundary = -W[0,0]/W[1,0]*xs + (0.5 - b[0])/W[1,0]   # the line  w1*x+w2*y+b=0.5
plt.figure(figsize=(5,4))
plt.scatter(inputs[:,0], inputs[:,1], c=(pred>0.5), s=6, cmap="bwr", alpha=.5)
plt.plot(xs, boundary, "-k", lw=2); plt.title("learned decision boundary"); plt.show()


## 3. The Keras API — where you'll actually work

### Three ways to build a model (know all three)
1. **Sequential** — a plain stack of layers. 90% of contest use.
2. **Functional** — a graph; needed for multiple inputs/outputs or branches.
3. **Subclassing** — write a `Model` class with custom `call()`; max flexibility.

In [ ]:
from keras import layers

# (1) Sequential
m1 = keras.Sequential([
    layers.Dense(64, activation="relu"),
    layers.Dense(10, activation="softmax"),
])

# (2) Functional — same model, explicit wiring
inp = keras.Input(shape=(784,))
x   = layers.Dense(64, activation="relu")(inp)
out = layers.Dense(10, activation="softmax")(x)
m2  = keras.Model(inp, out)

# (3) Subclassing
class MyModel(keras.Model):
    def __init__(self):
        super().__init__()
        self.d1 = layers.Dense(64, activation="relu")
        self.d2 = layers.Dense(10, activation="softmax")
    def call(self, x):
        return self.d2(self.d1(x))
m3 = MyModel()

m2.summary()   # build-time graph -> summary works immediately for functional


### compile → fit → evaluate → predict
`compile` wires the **optimizer**, **loss**, **metrics**. Strings are shortcuts for default objects;
pass objects when you need to set hyperparameters (e.g. learning rate).

In [ ]:
# string form (quick)
m1.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

# object form (control lr etc.) — equivalent but tunable
m1.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=[keras.metrics.SparseCategoricalAccuracy()],
)
print("compiled OK")


In [ ]:
# Train on MNIST to show fit/evaluate/predict + validation + callbacks
from keras.datasets import mnist
(Xtr,ytr),(Xte,yte) = mnist.load_data()
Xtr = Xtr.reshape(-1,784).astype("float32")/255
Xte = Xte.reshape(-1,784).astype("float32")/255

cbs = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=2,
                                     restore_best_weights=True)]
hist = m1.fit(Xtr, ytr, epochs=20, batch_size=128,
              validation_split=0.2, callbacks=cbs, verbose=0)
print("epochs actually run:", len(hist.history["loss"]))
print("test:", {k: round(v,4) for k,v in m1.evaluate(Xte, yte, verbose=0, return_dict=True).items()})
print("pred[:5]:", m1.predict(Xte[:5], verbose=0).argmax(1), "true:", yte[:5])


In [ ]:
# history is a dict of per-epoch values -> plot learning curves
import matplotlib.pyplot as plt
plt.plot(hist.history["loss"], label="train loss")
plt.plot(hist.history["val_loss"], label="val loss")
plt.xlabel("epoch"); plt.legend(); plt.title("learning curves"); plt.show()


## 4. PyTorch & JAX — 30-second awareness (you'll use Keras+TF, but don't get blindsided)

| | TensorFlow | PyTorch | JAX |
|---|---|---|---|
| trainable state | `tf.Variable` | `tensor(requires_grad=True)` / `nn.Parameter` | pytrees of arrays (functional) |
| autodiff | `tf.GradientTape` | `loss.backward()` then `.grad` | `jax.grad(fn)` |
| speed-up | `@tf.function` | `torch.compile` | `jax.jit` |
| style | graph/eager | eager, OOP | functional, pure |

**The point of Keras:** the *same* `model.fit(...)` code runs on any of these — you set the backend and
forget about these differences. For the contest, keep `KERAS_BACKEND="tensorflow"`.

---
# ✍️ PROBLEMS

### P1 — GradientTape by hand
For `f(x) = x**3 + 2*x`, use a `GradientTape` to compute f'(x) at x=2 (check: 3x²+2 = 14).
Then use nested tapes to get f''(2) (check: 6x = 12).

In [ ]:
import tensorflow as tf
# TODO


### P2 — Pure-TF linear *regression*
Generate `y = 3*x - 2 + noise` for 500 points. Using only `tf.Variable` + `GradientTape` (no Keras model),
fit `w, b` with a manual training loop and recover w≈3, b≈-2. Plot data + fitted line.

In [ ]:
# TODO


### P3 — Same model, three ways
Build the **same** 2-hidden-layer classifier (128→64→10 softmax) as (a) Sequential, (b) Functional,
(c) a subclassed `keras.Model`. Compile all three identically, fit each 3 epochs on MNIST, and confirm
they reach similar accuracy. Which felt easiest to write?

In [ ]:
# TODO


### P4 — Callbacks
Re-train the MNIST model with **both** `EarlyStopping(patience=3)` and
`ModelCheckpoint("best.keras", save_best_only=True)`. After training, load the saved best model with
`keras.models.load_model("best.keras")` and evaluate it. Confirm it matches the restored weights.

In [ ]:
# TODO


---
# 📋 TEMPLATES

### T1 — Set backend + imports (always first)

In [ ]:
import os; os.environ["KERAS_BACKEND"] = "tensorflow"
import keras
from keras import layers
import numpy as np, matplotlib.pyplot as plt


### T2 — Build (Sequential) + compile + fit + evaluate

In [ ]:
model = keras.Sequential([
    layers.Dense(128, activation="relu"),
    layers.Dense(64,  activation="relu"),
    layers.Dense(NUM_CLASSES, activation="softmax"),
])
model.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
cbs = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint("best.keras", save_best_only=True, monitor="val_loss"),
]
hist = model.fit(X_train, y_train, validation_split=0.2,
                 epochs=50, batch_size=128, callbacks=cbs, verbose=2)
print("test:", model.evaluate(X_test, y_test, verbose=0, return_dict=True))


### T3 — Functional API (multi-input / branches)

In [ ]:
inp = keras.Input(shape=(N_FEATURES,))
x   = layers.Dense(64, activation="relu")(inp)
x   = layers.Dropout(0.3)(x)
out = layers.Dense(NUM_CLASSES, activation="softmax")(x)
model = keras.Model(inp, out)


### T4 — Custom training loop (TF) — when fit() isn't enough

In [ ]:
optimizer = keras.optimizers.Adam(1e-3)
loss_fn   = keras.losses.SparseCategoricalCrossentropy()

@tf.function
def train_step(x, y):
    with tf.GradientTape() as tape:
        preds = model(x, training=True)
        loss  = loss_fn(y, preds)
    grads = tape.gradient(loss, model.trainable_weights)
    optimizer.apply_gradients(zip(grads, model.trainable_weights))
    return loss

# for epoch in range(E):
#     for xb, yb in dataset:      # dataset yields batches
#         train_step(xb, yb)


### T5 — Save / load

In [ ]:
model.save("model.keras")
restored = keras.models.load_model("model.keras")


---
### ✅ Checklist
- [ ] Explain Keras-vs-backend and name the 3 things every backend provides.
- [ ] Use `tf.Variable` (assign/assign_sub) and `GradientTape` (incl. nested) confidently.
- [ ] Implement a pure-TF linear model with a manual training loop.
- [ ] Build a model 3 ways (Sequential / Functional / subclass) and `compile/fit/evaluate/predict`.
- [ ] Add EarlyStopping + ModelCheckpoint and save/load a model.

**Next: Chapter 4** — Classification & regression on real tabular/text data (the bread-and-butter contest
tasks): binary, multiclass, and regression end-to-end with proper validation. Say "Chapter 4".